![DB Academy](https://files.training.databricks.com/binder/prod_main/enable-trusted-insights-for-your-organization-with-genie-en_us-1.0.0/images/20261001T150649Z/Enable Trusted Insights for your Organization with Genie/Includes/images/db-academy.png)

# Demo - Add Context to the Genie Agent Knowledge Store
#### Duration - 35 Minutes

## Overview

In the previous notebook, you created a Genie Agent and explored it. It works, but it has no metadata, SQL logic, or instructions yet.

This notebook is where you make it reliable. You will:

1. Review the common ways a Genie Agent can **go wrong**.
2. **Benchmark** your agent to establish a baseline and see where an un-curated agent falls short.
3. **Curate the Knowledge Store**: add metadata, then SQL logic, example queries, and instructions.
4. **Re-run the benchmarks** to measure the improvement.

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Describe common Genie Agent failure modes.**
2. **Add benchmarks with ground truth SQL** and run them to establish a baseline.
3. **Describe the iterative curation process** and the Knowledge Store order of operations.
4. **Add table and column metadata** in Unity Catalog with `ALTER TABLE`.
5. **Map each failing benchmark** to the curation tool that fixes it.
6. **Re-run benchmarks** to measure the impact of curation.

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #c62828; font-size: 1.1em;">Prerequisites</strong>
<p style="margin: 8px 0 0 0; color: #333;">Complete the previous notebook, <strong>Create and Explore Your First Genie Agent</strong>, before proceeding. This notebook builds on the tables and baseline agent you created there.</p>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select a Serverless SQL Warehouse</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless SQL Warehouse (2X-Small is sufficient)**
  - Select **Compute** > **More** > **SQL Warehouse** > Select a SQL Warehouse you have access to.

**NOTE:** This notebook was **developed and tested using Serverless SQL Warehouse**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>

## A. Classroom Setup

1. Run the cell below to set your default catalog and schema.
    - This assumes you have already run the setup in the previous notebook and your tables exist in **labuser_YOUR_USER_NAME.genie_course_bakehouse**.

In [0]:
%run ./Includes/Classroom-Setup-creating-table-column-descriptions

## B. Establish a Baseline with Benchmarks

Before curating, you need a way to measure whether curation actually helps. **Benchmarks** are provide that measure: a set of questions with known-correct answers you can re-run any time.

You will add **two benchmarks** to the agent you created in the previous notebook, then run them to establish a baseline. 

<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<style scoped>
.bench-header {
  background: #1B5162;
  color: white;
  border-radius: 10px 10px 4px 4px;
  padding: 22px 28px;
  display: flex;
  align-items: center;
  gap: 20px;
  margin-bottom: 14px;
}

.bench-header img {
  height: 56px;
  width: auto;
  flex-shrink: 0;
}

.bench-header-title {
  font-size: 22pt;
  font-weight: 700;
  line-height: 1.2;
}

.bench-header-sub {
  font-size: 15pt;
  opacity: 0.88;
  margin-top: 4px;
}

.bench-row {
  display: flex;
  gap: 14px;
  margin-bottom: 14px;
}

.bench-card {
  flex: 1;
  background: #F9F7F4;
  border-radius: 8px;
  padding: 20px 22px;
  position: relative;
  box-shadow: 0 2px 8px rgba(27,49,57,0.06);
  box-sizing: border-box;
}

.bench-card::before {
  content: "";
  position: absolute;
  top: 0;
  left: 0;
  width: 100%;
  height: 6px;
  border-radius: 8px 8px 0 0;
}

.bench-card.blue::before { background: #4299E0; }
.bench-card.green::before { background: #00A972; }
.bench-card.amber::before { background: #FFAB00; }

.bench-card-title {
  font-size: 16pt;
  font-weight: 700;
  color: #0b2026;
  margin-bottom: 12px;
}

.bench-card ul {
  margin: 0;
  padding-left: 20px;
  font-size: 14pt;
  color: #5A6F77;
  line-height: 1.6;
}

.bench-card li {
  margin-bottom: 10px;
}

.bench-card li:last-child {
  margin-bottom: 0;
}

.bench-ratings {
  background: #F9F7F4;
  border-radius: 8px;
  padding: 18px 22px;
  box-shadow: 0 2px 8px rgba(27,49,57,0.06);
  margin-bottom: 14px;
}

.bench-ratings-title {
  font-size: 15pt;
  font-weight: 700;
  color: #0b2026;
  margin-bottom: 12px;
}

.bench-ratings-row {
  display: flex;
  gap: 12px;
}

.bench-rating-pill {
  flex: 1;
  border-radius: 6px;
  padding: 12px 16px;
  font-size: 14pt;
  line-height: 1.5;
  color: #0b2026;
}

.bench-rating-pill.good {
  background: rgba(0,169,114,0.10);
  border-left: 4px solid #00A972;
}

.bench-rating-pill.bad {
  background: rgba(152,16,42,0.08);
  border-left: 4px solid #98102A;
}

.bench-rating-pill.manual {
  background: rgba(255,171,0,0.10);
  border-left: 4px solid #FFAB00;
}

.bench-rating-label {
  font-weight: 700;
  display: block;
  margin-bottom: 4px;
}

.bench-note {
  background: #F8F9FC;
  border-left: 4px solid #1B5162;
  border-radius: 6px;
  padding: 12px 18px;
  font-size: 14pt;
  color: #5A6F77;
  margin-bottom: 14px;
  line-height: 1.5;
}

.bench-link {
  display: block;
  margin-top: 12px;
  font-size: 14pt;
  color: #4299E0;
}
</style>

<!-- Header -->
<div class="bench-header">
  <div>
    <div class="bench-header-title">Benchmarks Overview</div>
    <div class="bench-header-sub">A set of test questions to assess Genie's overall response accuracy. Up to 500 per space.<br></br>
    Think of benchmarks as your <strong>acceptance criteria</strong>. They define what <strong>good</strong> looks like for your Genie Agent. As you iterate, re-run them to measure whether your changes actually improved accuracy.
    </div>
  </div>
</div>

<!-- Row 1: What a benchmark contains -->
<div class="bench-row">
  <div class="bench-card blue" style="flex: 1;">
    <div class="bench-card-title">Each Benchmark Contains</div>
    <ul>
      <li>A <strong>natural language question</strong> - phrased the way a real user would ask it</li>
      <li>An optional <strong>SQL Answer</strong> - the correct result used for automated accuracy scoring</li>
      <li style="background: rgba(66,153,224,0.12); border-radius: 6px; padding: 8px 10px; list-style: none; margin-left: -20px; padding-left: 30px;">Questions run as <strong>new conversations</strong> and no thread context is carried over</li>
    </ul>
  </div>
</div>

<!-- Row 2: Two purposes -->
<div class="bench-row">
  <div class="bench-card green">
    <div class="bench-card-title">Assess Accuracy</div>
    <ul>
      <li>Run benchmarks to evaluate Genie's response accuracy across your question set</li>
      <li>Include <strong>2-4 phrasings</strong> of the same question to test variation in how users ask it</li>
      <li>Only questions with a <strong>SQL Answer</strong> can be auto-scored. Others will require manual review</li>
    </ul>
  </div>
  <div class="bench-card amber">
    <div class="bench-card-title">Evaluate as You Refine</div>
    <ul>
      <li>Run benchmarks regularly as you add instructions, metadata, and example SQL</li>
      <li>Re-run a <strong>subset of questions</strong> from a previous result to test specific improvements</li>
      <li>Track accuracy over time using the timestamped <strong>Evaluations tab</strong></li>
    </ul>
  </div>
</div>

<!-- Ratings -->
<div class="bench-ratings">
  <div class="bench-ratings-title">How Responses Are Rated</div>
  <div class="bench-ratings-row">
    <div class="bench-rating-pill good">
      <span class="bench-rating-label" style="color: #00A972;">Good</span>
      <ul style="margin: 0; padding-left: 18px; font-size: 14pt; color: #0b2026; line-height: 1.6;">
        <li style="margin-bottom: 8px;">SQL exactly matches ground truth</li>
        <li style="margin-bottom: 8px;">Result sets match exactly</li>
        <li style="margin-bottom: 8px;">Results match, different sort order</li>
        <li>Numeric values round to same 4 significant digits</li>
      </ul>
    </div>
    <div class="bench-rating-pill bad">
      <span class="bench-rating-label" style="color: #98102A;">Bad</span>
      <ul style="margin: 0; padding-left: 18px; font-size: 14pt; color: #0b2026; line-height: 1.6;">
        <li style="margin-bottom: 8px;">Empty result set or error</li>
        <li style="margin-bottom: 8px;">Extra columns returned</li>
        <li>Single-cell result differs from ground truth</li>
      </ul>
    </div>
    <div class="bench-rating-pill manual">
      <span class="bench-rating-label" style="color: #b07d00;">Manual Review</span>
      <ul style="margin: 0; padding-left: 18px; font-size: 14pt; color: #0b2026; line-height: 1.6;">
        <li style="margin-bottom: 8px;">No SQL Answer provided</li>
        <li style="margin-bottom: 8px;">Results don't match closely enough for auto-scoring</li>
        <li>A human must mark Good or Bad</li>
      </ul>
    </div>
  </div>
</div>

<!-- Retention note -->
<div class="bench-note">
  <strong>Note:</strong> Evaluation response results are visible for <strong>one week</strong>. After that, results are no longer visible, but the generated SQL and example SQL statements remain.
</div>



</div>

Use benchmarks in a Genie Agent - 
[AWS](https://docs.databricks.com/aws/en/genie/benchmarks) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/genie-agents/monitor) |
[GCP](https://docs.databricks.com/gcp/en/genie/benchmarks)


### B1. Add the Number of Customers Benchmark

1. Open your **Genie Agent** from the previous notebook and select **Chat**.

2. Complete one of the following:
    
    a. If the last conversation you had is still available, move to **Step 3**.

    b. If not, ask the **Genie Agent**: <span style='background-color:#d4edda; padding:4px 6px; border-radius:4px;'><b>How many customers do we have?</b></span> and confirm the result is **300**.

3. In the response:
    - Select the more options button **⋮**
    - Select **Add as a benchmark**
    - **Add benchmark**

This captures your first benchmark. A simple, unambiguous question Genie should always get right. As the developer you can add it as you are testing the **Genie Agent**.

### B2. Add the Region Benchmark

Adding benchmarks using the UI.

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Why This Benchmark?
  </strong>
  <div style="color:#333;">

- This tests whether Genie knows our **business-defined regions**. 

- There is no `region` column in the data. 

- Our business groups countries into three regions: 
  - **APJ** (Japan, Australia)
  - **AMER** (US, Canada)
  - **EMEA** (Netherlands, France, Germany, Italy, Sweden). 

#### Without a SQL expression or instruction defining this mapping, Genie will likely group by `country` instead, or invent its own region definitions.

  </div>
</div>

1. Run the cell below to verify the correct result.

    Confirm the results show 3 regions: **APJ**, **AMER**, and **EMEA**.

In [0]:
SELECT
    CASE
        WHEN f.country IN ('Japan', 'Australia') THEN 'APJ'
        WHEN f.country IN ('US', 'Canada') THEN 'AMER'
        WHEN f.country IN ('Netherlands', 'France', 'Germany', 'Italy', 'Sweden') THEN 'EMEA'
        ELSE 'Other'
    END AS region,
    SUM(t.totalPrice) AS total_sales
FROM sales_franchises_gold AS f
JOIN sales_transactions_gold AS t
    ON f.franchiseID = t.franchiseID
GROUP BY
    region
ORDER BY
    total_sales DESC

2. In the Genie Agent, select **Benchmark** > **+ Add benchmark**.

3. Enter the question: `What are total sales by region?`

4. Run the cell below to generate the **Ground truth SQL answer (optional)** with your catalog name.
    - You can also select **Generate SQL** to use Genie Code to generate the SQL for this question. You can then review the SQL and use that as the ground truth.

In [0]:
DECLARE OR REPLACE my_query STRING DEFAULT "";

SET VAR my_query = "
SELECT
    CASE
        WHEN f.country IN ('Japan', 'Australia') THEN 'APJ'
        WHEN f.country IN ('US', 'Canada') THEN 'AMER'
        WHEN f.country IN ('Netherlands', 'France', 'Germany', 'Italy', 'Sweden') THEN 'EMEA'
        ELSE 'Other'
    END AS region,
    SUM(t.totalPrice) AS total_sales
FROM " || my_catalog || ".genie_course_bakehouse.sales_franchises_gold AS f
JOIN " || my_catalog || ".genie_course_bakehouse.sales_transactions_gold AS t
    ON f.franchiseID = t.franchiseID
GROUP BY region
ORDER BY total_sales DESC
";

SELECT my_query;

5. Copy the output above and paste into the **Ground truth SQL answer** field.

6. Select **Add benchmark**.

7. In the UI you should now see your two benchmarks.
    - *What are total sales by region?*
    - *How many customers do we have?*

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    This is Your Starting Point
  </strong>
  <div style="color:#333;">

These benchmarks represent your **baseline**. The questions Genie needs to answer correctly for this space to be reliable and consistent.

Right now, the **region** benchmark will most likely fail because the Genie Agent has **no metadata, no instructions, and no example SQL logic (Queries, Joins or Expressions)**.

As you work through the rest of this notebook, you will add context and re-run these benchmarks to measure progress.

#### In production, it is important to collaborate with **subject matter experts** to define your benchmark questions and ground truth answers. 

#### We have limited, simple benchmarks for training purposes.

  </div>
</div>

## C. Run the Baseline Benchmarks

### C1. Navigate to the Benchmarks Tab

1. In your Genie Agent, select **Benchmarks** at the top of the settings panel.

2. Select **Questions** and confirm you see both benchmarks:

   | # | Benchmark Question | Ground Truth SQL |
   |---|-------------------|-----------------|
   | 1 | How many customers do we have? | Yes |
   | 2 | What are total sales by region? | Yes |

### C2. Run All Benchmarks
<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Please Select Chat Mode</strong>
  <div style="color:#333;">

Please select the **Chat** button before running all benchmarks. We will be testing the **Chat** mode benchmarks in this course.

#### Benchmark questions support two modes:

- **Chat mode**: The default mode. Genie assesses accuracy by comparing its SQL-generated results against a provided SQL answer.

- **Agent mode**: Executes benchmark questions using the same multi-step reasoning as Genie's Agent mode. An LLM judge grades the responses. You can provide an optional evaluation note to guide grading.

Benchmarks Chat Mode and Agent Mode details: [AWS](https://docs.databricks.com/aws/en/genie-agents/monitor#benchmarks) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/genie-agents/monitor#benchmarks) |
[GCP](https://docs.databricks.com/gcp/en/genie-agents/monitor#benchmarks)
  </div>
</div>

1. Select **Chat** mode.

2. Select **Run all benchmarks** to execute all benchmark questions.

3. As the benchmarks are executing, continue on in this notebook. This may take a few minutes depending on the SQL warehouse.

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Important: Benchmarks Run as New Conversations
  </strong>
  <div style="color:#333;">

Each benchmark question is processed as a **brand new conversation**. 

This means:

- Genie does **not** carry over context from previous questions in the thread.
- The only context Genie uses is the **space configuration**: table schemas, instructions, example SQL, and SQL expressions.
- A question that worked correctly in your chat thread may **fail** as a benchmark if it relied on context from earlier questions.

#### This is by design. Benchmarks test how well your **agent configuration** supports each question independently.

  </div>
</div>

### C3. Review Benchmark Results

1. As a benchmark completes, review the results for each question.

2. For each benchmark, Genie shows:
   - **Assessment** - Whether the benchmark passed or failed
      - **Review proposed fixes (Knowledge Mining, explore but leave as is)** - It analyzes a failed benchmark response to identify likely issues and suggests changes to the Genie Agent's instructions or context. You can review, accept, or reject each proposed fix before applying it. 
         - We will learn how to add context manually first. 
         - Knowledge mining recommendations documentation - [AWS](https://docs.databricks.com/aws/en/genie-agents/tune-quality#knowledge-mining-recommendations) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/genie-agents/tune-quality#knowledge-mining) | [GCP](https://docs.databricks.com/gcp/en/genie-agents/tune-quality#knowledge-mining-recommendations)
      - For the purpose of the training we will manually add some of the suggestions found in the **Review knowledge snippets**. However, once you understand **Joins**, **SQL Expressions** and **SQL Queries** using the suggested snippets is extremely beneficial.
   - **Score reason** - Reason for the pass or fail
   - **Question** - Question that was asked
   - **Failure analysis** - Detailed reasoning on why the benchmark failed with the **SQL Differences**
      - **Response** - The SQL that Genie produced for this run
      - **Ground truth SQL answer** - If you provided ground truth SQL, Genie compares the **model output against the expected results (ground truth)**

3. Review the details of each result and note which benchmarks passed and which failed.
   - Specifically focus on the **why** these failed. 

4. Leave the benchmark page open.

**Review individual evaluations documentation** [AWS](https://docs.databricks.com/aws/en/genie-agents/monitor#review-individual-evaluations) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/genie-agents/monitor#review-individual-evaluations) |
[GCP](https://docs.databricks.com/gcp/en/genie-agents/monitor#review-individual-evaluations)

The table below summarizes expected behavior at this stage **before any metadata, SQL logic or instructions have been added**:

#### The results might vary. Our goal is to add enough context where we see correct and consistent Genie Agent results.

  <table style="width: 100%; border-collapse: collapse; font-size: 11pt;">
    <thead>
      <tr style="background: #1B5162; color: white;">
        <th style="padding: 10px; text-align: center; width: 40px;">#</th>
        <th style="padding: 10px; text-align: left;">Benchmark Question</th>
        <th style="padding: 10px; text-align: left;">Expected Behavior</th>
        <th style="padding: 10px; text-align: left;">Likely Result</th>
      </tr>
    </thead>
    <tbody>
      <tr style="background: #d4edda;">
        <td style="padding: 10px; text-align: center;">1</td>
        <td style="padding: 10px;"><em>How many customers do we have?</em></td>
        <td style="padding: 10px;">Simple COUNT on <code>sales_customers_gold</code></td>
        <td style="padding: 10px;"><strong>Should pass</strong>. Simple and unambiguous. If the ground truth saved earlier captured extra columns, you may need to update it.</td>
      </tr>
      <tr style="background: #f8d7da;">
        <td style="padding: 10px; text-align: center;">2</td>
        <td style="padding: 10px;"><em>What are total sales by region?</em></td>
        <td style="padding: 10px;">CASE expression mapping countries to APJ/AMER/EMEA</td>
        <td style="padding: 10px;"><strong>Likely fail</strong>. No region column exists. The Genie agent will most likely group by country</td>
      </tr>
    </tbody>
  </table>


<div style="border-left: 4px solid #FF5F46; background: #FFF6F4; padding: 14px 18px; border-radius: 4px; margin: 20px 0;">
  <strong style="display:block; color:#0b2026; margin-bottom:6px; font-size: 1.1em;">Two Benchmarks Is Just the Start</strong>
  <div style="color:#0b2026;">

We kept this demo to **two benchmarks** to stay focused. A real benchmark suite has many more.

As you build a Genie Agent, keep adding benchmarks that reflect the questions your users actually ask: multi-table joins, ambiguous business terms, data-quality edge cases, and business-defined groupings. In production, work with **subject matter experts** to define both the questions and the ground truth answers.

Then re-run the suite every time you add metadata, SQL logic, or instructions to confirm accuracy is actually improving. Later in this notebook you will curate this agent and re-run these benchmarks to confirm the improvement.
  </div>
</div>

## D. The Curation Mindset

### D1. The Iterative Curation Loop Overview
Effective Genie Agents are built through <b>iteration</b>, not perfection on the first try. Start small, test with real questions, and improve based on what you observe.

<br></br>
<!-- FRAME HEADER LABEL -->
<div style="background: #1B5162; color: white; padding: 10px 24px; font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; text-align: center; border-radius: 10px 10px 0 0;">Iterative Curation Loop</div>

<!-- LOOP FRAME -->
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 10px 10px; background: #F8F9FC; padding: 28px 32px 16px 32px;">
<!-- 4-STEP FLOW (NO BLANK LINES INSIDE THE GRID) -->
<div style="display: grid; grid-template-columns: 1fr auto 1fr auto 1fr auto 1fr; gap: 0; align-items: stretch;">
  <div style="background: #FFFFFF; border: 3px solid #4299E0; border-radius: 10px; padding: 16px 14px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center;">
    <div style="font-size: 13pt; font-weight: 700; color: #4299E0; letter-spacing: 1px; text-transform: uppercase; margin-bottom: 6px;">Step 1</div>
    <div style="font-size: 17pt; font-weight: 700; color: #0b2026; margin-bottom: 10px;">Create</div>
    <div style="font-size: 14pt; line-height: 1.4; color: #0b2026;">Add tables and metadata</div>
  </div>
  <div style="display: flex; align-items: center; justify-content: center; color: #618794; font-size: 26pt; line-height: 1; padding: 0 10px;">&rarr;</div>
  <div style="background: #FFFFFF; border: 3px solid #00A972; border-radius: 10px; padding: 16px 14px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center;">
    <div style="font-size: 13pt; font-weight: 700; color: #00A972; letter-spacing: 1px; text-transform: uppercase; margin-bottom: 6px;">Step 2</div>
    <div style="font-size: 17pt; font-weight: 700; color: #0b2026; margin-bottom: 10px;">Test Benchmarks</div>
    <div style="font-size: 14pt; line-height: 1.4; color: #0b2026;">Ask questions you expect users to ask</div>
  </div>
  <div style="display: flex; align-items: center; justify-content: center; color: #618794; font-size: 26pt; line-height: 1; padding: 0 10px;">&rarr;</div>
  <div style="background: #FFFFFF; border: 3px solid #FFAB00; border-radius: 10px; padding: 16px 14px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center;">
    <div style="font-size: 13pt; font-weight: 700; color: #FFAB00; letter-spacing: 1px; text-transform: uppercase; margin-bottom: 6px;">Step 3</div>
    <div style="font-size: 17pt; font-weight: 700; color: #0b2026; margin-bottom: 10px;">Observe</div>
    <div style="font-size: 14pt; line-height: 1.4; color: #0b2026;">Review generated SQL and identify gaps</div>
  </div>
  <div style="display: flex; align-items: center; justify-content: center; color: #618794; font-size: 26pt; line-height: 1; padding: 0 10px;">&rarr;</div>
  <div style="background: #FFFFFF; border: 3px solid #FF5F46; border-radius: 10px; padding: 16px 14px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center;">
    <div style="font-size: 13pt; font-weight: 700; color: #FF5F46; letter-spacing: 1px; text-transform: uppercase; margin-bottom: 6px;">Step 4</div>
    <div style="font-size: 17pt; font-weight: 700; color: #0b2026; margin-bottom: 10px;">Improve</div>
    <div style="font-size: 14pt; line-height: 1.4; color: #0b2026;">Add descriptions, instructions, or example SQL</div>
  </div>
</div>
<!-- LOOP-BACK U-SHAPE (NO BLANK LINES INSIDE) -->
<div style="position: relative; height: 80px; margin-top: 12px;">
  <div style="position: absolute; left: 37%; right: 13%; top: 0; height: 52px; border-left: 3px dashed #98102A; border-right: 3px dashed #98102A; border-bottom: 3px dashed #98102A; border-bottom-left-radius: 14px; border-bottom-right-radius: 14px;"></div>
  <div style="position: absolute; left: calc(37% - 7px); top: -6px; width: 0; height: 0; border-left: 8px solid transparent; border-right: 8px solid transparent; border-bottom: 11px solid #98102A;"></div>
  <div style="position: absolute; left: 50%; top: 42px; transform: translateX(-50%); background: #F8F9FC; padding: 2px 14px; color: #98102A; font-weight: 700; font-size: 14pt;">iterate</div>
</div>
</div>

<!-- BOTTOM TAKEAWAY -->
<div style="margin-top: 20px; padding: 14px 20px; background: #FFF6F4; border-left: 4px solid #FF5F46; border-radius: 6px; font-size: 14pt; line-height: 1.5;">
  <b style="color: #B83A24;">Key idea:</b> The loop never really ends. Every real user question is a chance to improve the Agent, and small, frequent passes beat large rewrites.
</div>

</div>

## E. Improving the Genie Agent with the Knowledge Store

### E1. The Knowledge Store and Order of Operations
The **Knowledge Store** is where you configure how Genie understands your data and responds to questions. This collection of curated semantic definitions **improves Genie's understanding of your data and increases response accuracy**.

Curate **iteratively**, but start with the highest-impact foundations first. Steps 2-6 are all configured through the Knowledge Store (via **Configure** in your Genie Agent).

<br></br>

<div style="max-width: 900px; margin: 0 auto 22px auto;">
<style>
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); font-family: sans-serif; box-sizing: border-box; }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.info { --bnr-bg: #1B5162; --bnr-accent: #4299E0; --bnr-icon-fg: #FFFFFF; --bnr-title: #FFFFFF; --bnr-tagline: #F9F7F4; }
</style>
<div class="ds-banner info">
  <div class="ds-banner-icon">i</div>
  <div>
    <div class="ds-banner-title">The Knowledge Store</div>
    <div class="ds-banner-tagline">Where you configure how Genie understands your data. Curate iteratively, starting with the highest-impact foundations.</div>
  </div>
</div>
</div>

<div class="rc-wrap">
<style>
.rc-wrap { width: 100%; display: flex; flex-direction: column; align-items: center; gap: 18px; font-family: sans-serif; }
.rc-wrap * { box-sizing: border-box; }
.rc-hint { width: 100%; max-width: 900px; text-align: center; font-size: 14pt; color: #5E7077; font-style: italic; }
.rc-hint strong { color: #FF5F46; font-style: normal; font-weight: 700; }
.rc-row { display: flex; flex-direction: column; gap: 12px; width: 100%; max-width: 900px; }
.rc-item { width: 100%; }
.rc-box { display: flex; align-items: center; gap: 18px; width: 100%; background: #F9F7F4; border: 2px solid transparent; border-left: 8px solid var(--pc); border-radius: 8px; padding: 16px 22px; text-align: left; cursor: pointer; user-select: none; transition: transform 0.12s, box-shadow 0.12s; box-shadow: 0 2px 8px rgba(27,49,57,0.06); }
.rc-box:hover { transform: translateX(3px); box-shadow: 0 4px 12px rgba(27,49,57,0.10); }
.rc-box.active { background: #fff; border-top-color: var(--pc); border-right-color: var(--pc); border-bottom-color: var(--pc); }
.rc-num { flex-shrink: 0; width: 40px; text-align: center; font-size: 24pt; font-weight: 800; color: var(--pc); line-height: 1; pointer-events: none; }
.rc-text { display: flex; flex-direction: column; gap: 4px; pointer-events: none; }
.rc-label { font-weight: 700; font-size: 15pt; letter-spacing: 1px; text-transform: uppercase; line-height: 1.2; }
.rc-title { font-size: 14pt; color: #0b2026; line-height: 1.3; }
.rc-detail-wrap { overflow: hidden; max-height: 0; opacity: 0; transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease; margin-top: 0; }
.rc-detail-wrap.open { max-height: 800px; opacity: 1; margin-top: 8px; }
.rc-detail-card { background: #FFFFFF; border-radius: 10px; padding: 20px 24px; border-left: 8px solid var(--pc); box-shadow: 0 2px 8px rgba(27,49,57,0.06); }
.rc-detail-title { font-size: 16pt; font-weight: 700; margin-bottom: 10px; line-height: 1.3; }
.rc-detail-body { font-size: 14pt; color: #0b2026; line-height: 1.55; }
.rc-detail-body ul { margin: 0; padding-left: 22px; }
.rc-detail-body li { margin-bottom: 10px; }
.rc-detail-body li:last-child { margin-bottom: 0; }
.rc-detail-body li.rc-key { list-style: none; margin: 12px 0 0 -22px; padding: 10px 14px; border-radius: 6px; border-left: 4px solid var(--pc); font-weight: 600; }
</style>

<div class="rc-hint">A recommended order. <strong>Click each step</strong> to expand it.</div>

<div class="rc-row">

  <div class="rc-item">
    <span class="rc-box" data-id="0" onclick="rcSelect(0)" style="--pc:#1B3139;">
      <span class="rc-num">1</span>
      <span class="rc-text">
        <span class="rc-label" style="color:#1B3139;">Purpose &amp; Scope (Completed)</span>
        <span class="rc-title">Focused audience, topic, and key questions</span>
      </span>
    </span>
    <div class="rc-detail-wrap" id="rc-detail-0" style="--pc:#1B3139;">
      <div class="rc-detail-card">
        <div class="rc-detail-title" style="color:#1B3139;">1. Define Purpose and Scope</div>
        <div class="rc-detail-body"><ul>
          <li>Start with a <strong>focused audience</strong>, topic, and a small set of business questions.</li>
          <li>These questions become your <strong>sample questions</strong> and <strong>benchmarks</strong>.</li>
          <li class="rc-key" style="background: rgba(27,49,57,0.08);">This is the planning step, not a Knowledge Store component, but it <strong>shapes everything that follows</strong>.</li>
        </ul></div>
      </div>
    </div>
  </div>

  <div class="rc-item">
    <span class="rc-box" data-id="1" onclick="rcSelect(1)" style="--pc:#4299E0;">
      <span class="rc-num">2</span>
      <span class="rc-text">
        <span class="rc-label" style="color:#4299E0;">Focused Tables (Completed)</span>
        <span class="rc-title">A small, well-documented set of tables</span>
      </span>
    </span>
    <div class="rc-detail-wrap" id="rc-detail-1" style="--pc:#4299E0;">
      <div class="rc-detail-card">
        <div class="rc-detail-title" style="color:#4299E0;">2. Add Focused Unity Catalog Tables</div>
        <div class="rc-detail-body"><ul>
          <li>Pick a <strong>small, well-documented</strong> set of tables or views (aim for <strong>5 or fewer</strong>).</li>
          <li>Prefer analytics-ready <strong>gold</strong> tables; pre-join with a view when the model is complex.</li>
          <li>Define <strong>foreign-key relationships</strong> in Unity Catalog when possible. You can also define joins explicitly in the Knowledge Store (see step 4).</li>
          <li class="rc-key" style="background: rgba(66,153,224,0.12);">Get the <strong>foundation right first</strong>: the wrong tables cannot be fixed by downstream curation.</li>
        </ul></div>
      </div>
    </div>
  </div>

  <div class="rc-item">
    <span class="rc-box" data-id="2" onclick="rcSelect(2)" style="--pc:#98102A;">
      <span class="rc-num">3</span>
      <span class="rc-text">
        <span class="rc-label" style="color:#98102A;">Metadata &amp; Prompt Matching</span>
        <span class="rc-title">Strongest early accuracy lever</span>
      </span>
    </span>
    <div class="rc-detail-wrap" id="rc-detail-2" style="--pc:#98102A;">
      <div class="rc-detail-card">
        <div class="rc-detail-title" style="color:#98102A;">3. Add Metadata and Prompt Matching</div>
        <div class="rc-detail-body"><ul>
          <li><strong>Space-Level Metadata:</strong> descriptions for tables and columns, plus synonyms aligned to business language. Extends or overrides Unity Catalog metadata with no changes to the data.</li>
          <li><strong>Prompt Matching:</strong> format assistance provides representative column values; entity matching maps user language to real data values. Improves categorical filters.</li>
          <li class="rc-key" style="background: rgba(152,16,42,0.08);">This is the <strong>strongest early accuracy lever</strong>, and often the easiest win.</li>
        </ul></div>
      </div>
    </div>
  </div>

  <div class="rc-item">
    <span class="rc-box" data-id="3" onclick="rcSelect(3)" style="--pc:#00A972;">
      <span class="rc-num">4</span>
      <span class="rc-text">
        <span class="rc-label" style="color:#00A972;">SQL Logic</span>
        <span class="rc-title">Joins plus reusable metrics, filters, and dimensions</span>
      </span>
    </span>
    <div class="rc-detail-wrap" id="rc-detail-3" style="--pc:#00A972;">
      <div class="rc-detail-card">
        <div class="rc-detail-title" style="color:#00A972;">4. Add SQL Logic (Joins and Expressions)</div>
        <div class="rc-detail-body"><ul>
          <li><strong>Joins:</strong> define how tables relate, specify join conditions explicitly, and set relationship types (one-to-many, many-to-one, one-to-one).</li>
          <li><strong>Expressions:</strong> reusable <strong>measures and KPIs</strong>, <strong>filters</strong>, and <strong>dimensions</strong> (grouping attributes).</li>
          <li>Each expression can include <strong>synonyms</strong> for natural language queries.</li>
          <li class="rc-key" style="background: rgba(0,169,114,0.10);">This is the <strong>preferred way</strong> to encode business definitions, rather than text instructions.</li>
        </ul></div>
      </div>
    </div>
  </div>

  <div class="rc-item">
    <span class="rc-box" data-id="4" onclick="rcSelect(4)" style="--pc:#FFAB00;">
      <span class="rc-num">5</span>
      <span class="rc-text">
        <span class="rc-label" style="color:#B07D00;">Example SQL Queries</span>
        <span class="rc-title">Verified question-to-SQL patterns</span>
      </span>
    </span>
    <div class="rc-detail-wrap" id="rc-detail-4" style="--pc:#FFAB00;">
      <div class="rc-detail-card">
        <div class="rc-detail-title" style="color:#B07D00;">5. Add Example SQL Queries</div>
        <div class="rc-detail-body"><ul>
          <li>Pair a <strong>user question</strong> with the SQL that answers it.</li>
          <li>Teaches Genie common or complex <strong>question patterns</strong>; can be <strong>parameterized</strong> and reused.</li>
          <li><strong>SQL functions</strong> can be registered in Unity Catalog for reusable, trusted logic across teams.</li>
          <li class="rc-key" style="background: rgba(255,171,0,0.16);">One of the <strong>most effective levers</strong> for accurate, consistent answers.</li>
        </ul></div>
      </div>
    </div>
  </div>

  <div class="rc-item">
    <span class="rc-box" data-id="5" onclick="rcSelect(5)" style="--pc:#FF5F46;">
      <span class="rc-num">6</span>
      <span class="rc-text">
        <span class="rc-label" style="color:#FF5F46;">General Instructions</span>
        <span class="rc-title">Global text rules, as a last resort</span>
      </span>
    </span>
    <div class="rc-detail-wrap" id="rc-detail-5" style="--pc:#FF5F46;">
      <div class="rc-detail-card">
        <div class="rc-detail-title" style="color:#FF5F46;">6. Add General Instructions</div>
        <div class="rc-detail-body"><ul>
          <li><strong>Global text guidance</strong> applied to all Genie responses.</li>
          <li>Capture <strong>business context</strong> and domain-specific rules.</li>
          <li>Control <strong>formatting and behavior</strong> (language, rounding, output style).</li>
          <li class="rc-key" style="background: rgba(255,95,70,0.10);"><strong>Last resort:</strong> use only for logic that cannot be expressed in SQL or metadata. Avoid defining metrics or joins here.</li>
        </ul></div>
      </div>
    </div>
  </div>

</div>
</div>

<script>
var rcCurrent = null;
function rcSelect(id) {
  document.querySelectorAll('.rc-box').forEach(function(b) {
    b.classList.toggle('active', parseInt(b.dataset.id) === id && rcCurrent !== id);
  });
  document.querySelectorAll('.rc-detail-wrap').forEach(function(w) { w.classList.remove('open'); });
  if (rcCurrent === id) { rcCurrent = null; return; }
  rcCurrent = id;
  document.getElementById('rc-detail-' + id).classList.add('open');
}
</script>

<div style="max-width: 900px; margin: 16px auto 0 auto; padding: 14px 18px; background: #F8F9FC; border-left: 4px solid #1B5162; border-radius: 6px;">
  <div style="font-size: 15pt; color: #0b2026; line-height: 1.6;">
    <strong>Also available:</strong> Space-level data customization lets you hide columns and simplify datasets for users. This improves usability, but does not change how Genie understands questions or generates SQL.
  </div>
</div>

<div style="max-width: 900px; margin: 20px auto 0 auto; padding: 18px 24px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 16pt; color: #0b2026; line-height: 1.6;">
    <div style="font-weight: 700; margin-bottom: 8px;">Key points:</div>
    <ul style="padding-left: 20px; margin: 0;">
      <li>Configurations are <strong>scoped to a single Genie Agent</strong></li>
      <li><strong>No</strong> changes to Unity Catalog metadata</li>
      <li>Safe experimentation across <strong>multiple spaces</strong></li>
      <li>Shared data remains <strong>unchanged</strong></li>
    </ul>
  </div>
</div>
<!-- Docs callout -->
<div style="margin-top: 24px; padding: 16px 24px; background: #1B5162; border-radius: 8px; display: flex; align-items: center; justify-content: space-between; gap: 20px;">
  <div style="font-size: 15pt; color: white; line-height: 1.5;">
    For more information on the <strong>Knowledge Store</strong>, visit the official Databricks documentation.
  </div>
  <div style="display: flex; gap: 10px; flex-shrink: 0;">
    <a href="https://docs.databricks.com/aws/en/genie/knowledge-store" target="_blank" style="background: white; color: #1B5162; font-size: 14pt; font-weight: 700; padding: 10px 20px; border-radius: 6px; text-decoration: none; white-space: nowrap;">
      AWS →
    </a>
    <a href="https://learn.microsoft.com/en-us/azure/databricks/genie-agents/tune-quality" target="_blank" style="background: white; color: #1B5162; font-size: 14pt; font-weight: 700; padding: 10px 20px; border-radius: 6px; text-decoration: none; white-space: nowrap;">
      Azure →
    </a>
    <a href="https://docs.databricks.com/gcp/en/genie/knowledge-store" target="_blank" style="background: white; color: #1B5162; font-size: 14pt; font-weight: 700; padding: 10px 20px; border-radius: 6px; text-decoration: none; white-space: nowrap;">
      GCP →
    </a>
  </div>
</div>

</div>

##### EXPAND FOR ADDITIONAL NOTES

<details>

The Knowledge Store is the single surface where you configure how Genie interprets questions and generates SQL. The six clickable steps above are the **recommended order** to build it. Order matters, because each layer builds on the previous one.

#### Why order matters
- **1. Purpose and Scope** is planning, not configuration, but it defines the questions that everything else is optimized for. A space that tries to answer everything answers nothing well.
- **2. Focused Tables** are the foundation. No amount of downstream curation compensates for the wrong tables. Aim for 5 or fewer analytics-ready tables, and define foreign-key relationships in Unity Catalog.
- **3. Metadata and Prompt Matching** is the strongest early accuracy lever. Descriptions and synonyms map business language to your columns, and prompt matching connects user terms to real data values. Many spaces improve significantly here before any SQL is added.
- **4. SQL Logic (Joins and Expressions)** defines how tables relate and encodes reusable business logic (metrics, filters, dimensions). This is the preferred way to define business terms, rather than plain text.
- **5. Example SQL Queries** teach Genie question-to-SQL patterns for common or complex asks, and can be parameterized or registered as SQL functions.
- **6. General Instructions** are the last resort, for global rules that cannot be expressed any other way. Keep them short and specific, and never use them to define metrics or joins.

#### The priority, in one line
Prioritize **metadata** for meaning, then **SQL logic and queries** for business definitions, and use **instructions only when necessary**. Structured inputs drive accuracy; text is fallback guidance.

#### Scoped and safe
Everything in the Knowledge Store is scoped to a single Genie Agent. It never changes Unity Catalog metadata or the underlying data, so you can experiment freely across multiple spaces.

</details>

### E2. View the Genie Agent Knowledge Store

Follow the steps below to explore the configuration options available in your Genie Agent.

1. If necessary, open your **Genie Agent** (Right-click **Genie Agents** in the left navigation and select **Open Link in New Tab**).

2. Select **Configure**.

3. Select **Sources**
   - View available **Table and Column Metadata** (descriptions and synonyms)
   - Currently no metadata is set on the tables

4. Select **Instructions**
   - **Text**: Define **text instructions** for behavior and edge cases

5. Select **Examples > Add**
   - **Example query**: Add **example queries** that Genie uses as trusted patterns (SQL logic)
   - **Filter**: Add example filters and instructions to use
   - **Measure & Filed (SQL Expressions)**: Define **business metrics as SQL expressions** (SQL logic)
   - **Join**: Specify **relationships between tables** (SQL logic)



## F. Connecting Problems to Solutions

First, review the common problems any Genie Agent can run into. Then see how the specific problems from your bakehouse questions map to the exact curation tools that fix them.

### F1. Mapping Demo Problems to Curation Tools

Earlier, you observed that Genie can struggle with specific types of questions without the right context.

Depending on the problem, the solution maps directly to a curation tool.

<br></br>

<div style="font-family: sans-serif; color: #0b2026; max-width: 1400px; margin: 0 auto;">

<div style="display: flex; gap: 20px; justify-content: center; flex-wrap: nowrap;">

  <!-- Problem 1 -->
  <div style="flex: 1; border: 2px solid #e0e0e0; border-radius: 12px; overflow: hidden;">
    <div style="background: #2574B5; color: white; padding: 14px 20px; text-align: center;">
      <div style="font-size: 18pt; font-weight: bold;">Opaque Column Names and Values</div>
    </div>
    <div style="padding: 18px 20px;">
      <div style="font-size: 15pt; font-style: italic; color: #333; margin-bottom: 10px; text-align: center;">"Count of Bad Reviews by Location?"</div>
      <div style="font-size: 14pt; color: #555; line-height: 1.6; margin-bottom: 14px;">
        Genie might not know the valid values for the <strong>flag</strong> column. It may have guessed the wrong case, value, or skipped the column entirely.
      </div>
      <div style="background: rgba(37,116,181,0.10); border-left: 4px solid #2574B5; padding: 10px 12px; border-radius: 6px; font-size: 14pt;">
        <strong>Fix:</strong> Add a <strong>column description</strong> listing valid values: <code>positive</code>, <code>negative</code>, <code>mixed</code>
      </div>
    </div>
  </div>

  <!-- Problem 2 -->
  <div style="flex: 1; border: 2px solid #e0e0e0; border-radius: 12px; overflow: hidden;">
    <div style="background: #02A36F; color: white; padding: 14px 20px; text-align: center;">
      <div style="font-size: 18pt; font-weight: bold;">Ambiguous Business Terms</div>
    </div>
    <div style="padding: 18px 20px;">
      <div style="font-size: 15pt; font-style: italic; color: #333; margin-bottom: 10px; text-align: center;">"Who are our best customers?"</div>
      <div style="font-size: 14pt; color: #555; line-height: 1.6; margin-bottom: 14px;">
        Genie had to guess whether "best" meant highest spend, most orders, or most recent purchase. Different runs may give different answers.
      </div>
      <div style="background: rgba(2,163,111,0.10); border-left: 4px solid #02A36F; padding: 10px 12px; border-radius: 6px; font-size: 14pt;">
        <strong>Fix:</strong> Add a <strong>SQL query</strong>: "Define a question with an example query for the Genie Agent".
      </div>
    </div>
  </div>

  <!-- Problem 3 -->
  <div style="flex: 1; border: 2px solid #e0e0e0; border-radius: 12px; overflow: hidden;">
    <div style="background: #FE3722; color: white; padding: 14px 20px; text-align: center;">
      <div style="font-size: 18pt; font-weight: bold;">Complex Query Patterns</div>
    </div>
    <div style="padding: 18px 20px;">
      <div style="font-size: 15pt; font-style: italic; color: #333; margin-bottom: 10px; text-align: center;">"How Many Franchises Does Each Supplier Serve?"</div>
      <div style="font-size: 14pt; color: #555; line-height: 1.6; margin-bottom: 14px;">
        Genie struggled to generate the LEFT JOIN + IS NULL pattern needed to find orphaned foreign keys and aggregate all of the data.
      </div>
      <div style="background: rgba(254,55,34,0.10); border-left: 4px solid #FE3722; padding: 10px 12px; border-radius: 6px; font-size: 14pt;">
        <strong>Fix:</strong> Add an <strong>example SQL query</strong> and/or <strong>define a SQL JOIN</strong> demonstrating the LEFT JOIN pattern
      </div>
    </div>
  </div>

</div>

<!-- Docs callout -->
<div style="margin-top: 24px; padding: 16px 24px; background: #1B5162; border-radius: 8px; display: flex; align-items: center; justify-content: space-between; gap: 20px;">
  <div style="font-size: 15pt; color: white; line-height: 1.5;">
    For more common issues and troubleshooting guidance, visit the official Databricks documentation.
  </div>
  <div style="display: flex; gap: 10px; flex-shrink: 0;">
    <a href="https://docs.databricks.com/aws/en/genie/troubleshooting" target="_blank" style="background: white; color: #1B5162; font-size: 14pt; font-weight: 700; padding: 10px 20px; border-radius: 6px; text-decoration: none; white-space: nowrap;">
      AWS →
    </a>
    <a href="https://learn.microsoft.com/en-us/azure/databricks/genie-agents/troubleshooting" target="_blank" style="background: white; color: #1B5162; font-size: 14pt; font-weight: 700; padding: 10px 20px; border-radius: 6px; text-decoration: none; white-space: nowrap;">
      Azure →
    </a>
    <a href="https://docs.databricks.com/gcp/en/genie/troubleshooting" target="_blank" style="background: white; color: #1B5162; font-size: 14pt; font-weight: 700; padding: 10px 20px; border-radius: 6px; text-decoration: none; white-space: nowrap;">
      GCP →
    </a>
  </div>
</div>

</div>


### F2. What Could Go Wrong When Building a Genie Agent Overview

A Genie Agent improves through iteration. Understanding common failure modes helps create and fix them faster.

#### Click each issue to expand the recommended fix.

<br></br>

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">

<style>
.acc-item {
  border: 2px solid #e0e0e0;
  border-radius: 8px;
  margin-bottom: 8px;
  overflow: hidden;
  transition: border-color 0.2s;
}

.acc-item.open { border-color: #98102A; }

.acc-header {
  display: flex;
  align-items: center;
  gap: 14px;
  padding: 14px 18px;
  cursor: pointer;
  user-select: none;
  background: #F9F7F4;
  transition: background 0.15s;
}

.acc-item.open .acc-header { background: #FDECEA; }

.acc-num {
  width: 32px;
  height: 32px;
  border-radius: 50%;
  background: #e0e0e0;
  color: #666;
  font-size: 14pt;
  font-weight: 700;
  display: flex;
  align-items: center;
  justify-content: center;
  flex-shrink: 0;
  transition: background 0.2s, color 0.2s;
}

.acc-item.open .acc-num { background: #98102A; color: #fff; }

.acc-summary {
  flex: 1;
  font-size: 15pt;
  font-weight: 600;
  color: #0b2026;
}

.acc-chevron {
  font-size: 14pt;
  color: #888;
  transition: transform 0.25s;
}

.acc-item.open .acc-chevron { transform: rotate(180deg); color: #98102A; }

.acc-body {
  max-height: 0;
  opacity: 0;
  overflow: hidden;
  transition: max-height 0.3s ease, opacity 0.25s ease, padding 0.25s ease;
  padding: 0 18px 0 64px;
  font-size: 14pt;
  line-height: 1.7;
  color: #333;
}

.acc-item.open .acc-body {
  max-height: 600px;
  opacity: 1;
  padding: 4px 18px 18px 64px;
}

.acc-body ul {
  margin: 8px 0 0 0;
  padding-left: 20px;
}

.acc-body li {
  margin-bottom: 8px;
}

.acc-fix {
  margin-top: 12px;
  padding: 10px 14px;
  border-left: 4px solid #98102A;
  background: rgba(152,16,42,0.08);
  border-radius: 0 6px 6px 0;
  font-size: 14pt;
}
</style>

<div id="acc-container"></div>



</div>

<script>
var ACC_DATA = [
  {
    summary: 'Misunderstood Business Jargon',
    detail: 'Genie may misinterpret company-specific terms like fiscal <strong>year, region names, or product codes</strong>.',
    fix: '<strong>Fix:</strong> Add instructions that explicitly map your business jargon to concepts Genie understands. For example, define what "this year" means in terms of your fiscal calendar. See <em>Provide Instructions</em> in the Knowledge Store.'
  },
  {
    summary: 'Incorrect Table or Column Usage',
    detail: 'Genie is pulling data from the <strong>wrong table or querying the wrong column</strong>.',
    fix: '<strong>Fix:</strong> <ul><li>Add clear table and column descriptions that match the language users actually type</li><li>Provide example SQL queries showing the correct tables and columns to use</li><li>Remove or hide overlapping tables and columns that confuse Genie</li></ul>'
  },
  {
    summary: 'Filtering Errors (Wrong Values in WHERE Clause)',
    detail: 'Genie filters on the wrong value. <br><br>For example, using "California" when the data stores "CA".',
    fix: '<strong>Fix:</strong> Enable <strong>Example values</strong> and <strong>Value dictionaries</strong> on the relevant columns via Prompt Matching in the Knowledge Store. If new data has been added, refresh the values.'
  },
  {
    summary: 'Incorrect Joins',
    detail: 'Genie doesn\'t know how to join tables because foreign key relationships are not defined.',
    fix: '<strong>Fix:</strong> <ul><li>Define foreign key references in Unity Catalog using the CONSTRAINT clause</li><li>Add join relationships in the Knowledge Store for complex joins or when you can\'t modify the underlying tables</li><li>Provide example queries that demonstrate the correct join pattern</li><li>As a last resort, pre-join tables into a view and use that in the agent instead</li></ul>'
  },
  {
    summary: 'Metric Calculation Issues',
    detail: 'Genie computes metrics incorrectly. <br><br>Wrong aggregation, wrong roll-up, or inconsistent results across questions.',
    fix: '<strong>Fix:</strong> <ul><li>Define metrics as <strong>SQL expressions</strong> in the Knowledge Store for reusable, precise definitions</li><li>Provide example SQL queries showing how each metric is calculated</li><li>If metrics are pre-aggregated in a table, explain this in table comments and specify valid aggregations</li></ul>'
  },
  {
    summary: 'Shared or Ambiguous Column Names',
    detail: 'Two tables have a column with the same name. <br><br>For example, both <code>sales_customers</code> and <code>sales_franchises</code> have a <code>country</code> column. Genie may query the wrong one.',
    fix: '<strong>Fix:</strong> <ul><li>Add column descriptions that explicitly clarify which table\'s column represents what, e.g., "This is the <strong>customer\'s</strong> country, not the franchise location country"</li><li>Add an instruction or example SQL query that shows which <code>country</code> column to use for revenue or location questions</li><li>Hide the ambiguous column from the agent if it isn\'t needed for the target use case</li></ul>'
  }
];

(function() {
  var container = document.getElementById('acc-container');
  ACC_DATA.forEach(function(item, i) {
    var div = document.createElement('div');
    div.className = 'acc-item';
    div.innerHTML =
      '<div class="acc-header" onclick="toggleAcc(' + i + ')">'
      + '<div class="acc-num">' + (i + 1) + '</div>'
      + '<div class="acc-summary">' + item.summary + '</div>'
      + '<div class="acc-chevron">&#9660;</div>'
      + '</div>'
      + '<div class="acc-body">'
      + '<div style="margin-bottom:10px;">' + item.detail + '</div>'
      + '<div class="acc-fix">' + item.fix + '</div>'
      + '</div>';
    container.appendChild(div);
  });
})();

function toggleAcc(idx) {
  var items = document.querySelectorAll('.acc-item');
  items.forEach(function(el, i) {
    el.classList.toggle('open', i === idx && !el.classList.contains('open'));
  });
}
</script>

## G. Where We Are: The Authoring Checklist

This demo covers steps **3 through 6**: metadata and prompt matching, SQL logic, example SQL queries, and general instructions. You **test, benchmark, and iterate** throughout, and that loop wraps every step.

<br></br>

<div style="max-width: 1280px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<!-- 6-STEP FLOW -->
<div style="display: grid; grid-template-columns: 1fr auto 1fr auto 1fr auto 1fr auto 1fr auto 1fr; gap: 0; align-items: stretch;">

  <!-- STEP 1 (DIMMED) -->
  <div style="background: #EEEDE9; border: 3px solid #C2C2C2; border-radius: 10px; padding: 16px 12px; box-shadow: none; text-align: center; min-width: 0; opacity: 0.7;">
    <div style="font-size: 26pt; font-weight: 800; color: #90A5B1; line-height: 1; margin-bottom: 8px;">1</div>
    <div style="font-size: 14pt; font-weight: 600; color: #5A6F77; line-height: 1.35;">Define purpose<br>and scope</div>
  </div>

  <div style="display: flex; align-items: center; justify-content: center; color: #C2C2C2; font-size: 22pt; line-height: 1;">&rarr;</div>

  <!-- STEP 2 (DIMMED) -->
  <div style="background: #EEEDE9; border: 3px solid #C2C2C2; border-radius: 10px; padding: 16px 12px; box-shadow: none; text-align: center; min-width: 0; opacity: 0.7;">
    <div style="font-size: 26pt; font-weight: 800; color: #90A5B1; line-height: 1; margin-bottom: 8px;">2</div>
    <div style="font-size: 14pt; font-weight: 600; color: #5A6F77; line-height: 1.35;">Add focused<br>tables</div>
  </div>

  <div style="display: flex; align-items: center; justify-content: center; color: #618794; font-size: 22pt; line-height: 1;">&rarr;</div>

  <!-- STEP 3 (active) -->
  <div style="background: #F9F7F4; border: 3px solid #F8A805; border-radius: 10px; padding: 16px 12px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; min-width: 0;">
    <div style="font-size: 26pt; font-weight: 800; color: #F8A805; line-height: 1; margin-bottom: 8px;">3</div>
    <div style="font-size: 14pt; font-weight: 600; color: #0b2026; line-height: 1.35;">Add metadata and<br>prompt matching</div>
  </div>

  <div style="display: flex; align-items: center; justify-content: center; color: #618794; font-size: 22pt; line-height: 1;">&rarr;</div>

  <!-- STEP 4 (active) -->
  <div style="background: #F9F7F4; border: 3px solid #FF5F46; border-radius: 10px; padding: 16px 12px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; min-width: 0;">
    <div style="font-size: 26pt; font-weight: 800; color: #FF5F46; line-height: 1; margin-bottom: 8px;">4</div>
    <div style="font-size: 14pt; font-weight: 600; color: #0b2026; line-height: 1.35;">Add SQL logic<br>(joins, expressions)</div>
  </div>

  <div style="display: flex; align-items: center; justify-content: center; color: #618794; font-size: 22pt; line-height: 1;">&rarr;</div>

  <!-- STEP 5 (active) -->
  <div style="background: #F9F7F4; border: 3px solid #98182A; border-radius: 10px; padding: 16px 12px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; min-width: 0;">
    <div style="font-size: 26pt; font-weight: 800; color: #98182A; line-height: 1; margin-bottom: 8px;">5</div>
    <div style="font-size: 14pt; font-weight: 600; color: #0b2026; line-height: 1.35;">Add example<br>SQL queries</div>
  </div>

  <div style="display: flex; align-items: center; justify-content: center; color: #618794; font-size: 22pt; line-height: 1;">&rarr;</div>

  <!-- STEP 6 (active) -->
  <div style="background: #F9F7F4; border: 3px solid #1B3139; border-radius: 10px; padding: 16px 12px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; min-width: 0;">
    <div style="font-size: 26pt; font-weight: 800; color: #1B3139; line-height: 1; margin-bottom: 8px;">6</div>
    <div style="font-size: 14pt; font-weight: 600; color: #0b2026; line-height: 1.35;">Add general<br>instructions</div>
  </div>

</div>


<!-- WRAPPING LOOP: test/iterate wraps all six steps -->
<div style="width: 100%; max-width: 1280px; margin: 14px auto 0 auto; background: rgba(27,49,57,0.06); border: 2px dashed #618794; border-radius: 10px; padding: 12px 18px; display: flex; align-items: center; justify-content: center; gap: 10px; box-sizing: border-box;">
  <span style="font-size: 22pt; color: #1B5162; line-height: 1;">&#8635;</span>
  <span style="font-size: 14pt; font-weight: 700; color: #1B5162;">Test, benchmark, and iterate. Repeat after every step.</span>
</div>
</div>

## H. Add Table and Column Metadata (Step 3)

Metadata is the foundation and the easiest high-impact win. Rather than adding descriptions one by one in Catalog Explorer, you will use `ALTER TABLE` SQL statements to add all table and column descriptions and primary/foreign key constraints at once.

| What | Why |
|------|-----|
| **Table descriptions** (`COMMENT ON TABLE`) | Tells Genie what each table represents and when to use it |
| **Column descriptions** (`ALTER COLUMN ... COMMENT`) | Clarifies what each column means, lists valid values, and disambiguates shared column names |
| **Primary / foreign key constraints** (`ADD CONSTRAINT ... NOT ENFORCED`) | Informational metadata Genie reads to auto-detect join relationships |

These are stored directly in **Unity Catalog** and Genie reads them automatically. **No additional configuration is required.**

### H1. Source Column and Table Metadata in Unity Catalog

1. Run the cell below to add all descriptions and constraints at once. Feel free to explore the metadata as the code runs.

**NOTE:** You can also use **Genie Code** to help generate metadata for your tables and columns. Always review AI generated metadata for accuracy.
[AWS](https://docs.databricks.com/aws/en/genie-code/) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/genie-code/use-genie-code) |
[GCP](https://docs.databricks.com/gcp/en/genie-code/)

In [0]:
-- -----------------------------------------------------------
-- sales_customers_gold
-- -----------------------------------------------------------
COMMENT ON TABLE sales_customers_gold
  IS 'Customer profiles for the bakehouse franchise business. Each row represents one unique customer. There are no duplicate or null customerIDs. To count total customers, use COUNT(*) on this table. Use customerID to join with sales_transactions_gold.';

ALTER TABLE sales_customers_gold
  ALTER COLUMN customerID      COMMENT 'Unique identifier for each customer. Primary key. Every row has a non-null, unique value. Use COUNT(*) to count total customers. Join key to sales_transactions_gold.customerID.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN first_name      COMMENT 'Customer first name.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN last_name       COMMENT 'Customer last name.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN email_address   COMMENT 'Customer email address.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN phone_number    COMMENT 'Customer phone number.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN address         COMMENT 'Customer street address.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN city            COMMENT 'City where the customer lives. This is the customer city, not the franchise city.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN state           COMMENT 'State or province where the customer lives.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN country         COMMENT 'Country where the customer lives. This is the customer country, not the franchise country.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN continent       COMMENT 'Continent where the customer lives.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN postal_zip_code COMMENT 'Postal or zip code for the customer address.';
ALTER TABLE sales_customers_gold
  ALTER COLUMN gender          COMMENT 'Customer gender. Valid values: female, male.';

ALTER TABLE sales_customers_gold ALTER COLUMN customerID SET NOT NULL;
ALTER TABLE sales_customers_gold ADD CONSTRAINT pk_customers PRIMARY KEY (customerID) NOT ENFORCED;

-- -----------------------------------------------------------
-- sales_suppliers_gold  (must come before franchises so PK exists for FK reference)
-- -----------------------------------------------------------
COMMENT ON TABLE sales_suppliers_gold
  IS 'Ingredient suppliers for bakehouse franchises. Each row represents a supplier that provides a specific ingredient. Use supplierID to join with sales_franchises_gold.';

ALTER TABLE sales_suppliers_gold
  ALTER COLUMN supplierID   COMMENT 'Unique identifier for each supplier. Join key to sales_franchises_gold.supplierID.';
ALTER TABLE sales_suppliers_gold
  ALTER COLUMN name         COMMENT 'Name of the supplier company.';
ALTER TABLE sales_suppliers_gold
  ALTER COLUMN ingredient   COMMENT 'The specific ingredient this supplier provides (e.g., almonds, vanilla, cinnamon, honey).';
ALTER TABLE sales_suppliers_gold
  ALTER COLUMN continent    COMMENT 'Continent where the supplier is located.';
ALTER TABLE sales_suppliers_gold
  ALTER COLUMN city         COMMENT 'City where the supplier is located.';
ALTER TABLE sales_suppliers_gold
  ALTER COLUMN district     COMMENT 'District or neighborhood where the supplier is located.';
ALTER TABLE sales_suppliers_gold
  ALTER COLUMN size         COMMENT 'Size category of the supplier. Valid values: S (Small), M (Medium), L (Large), XL (Extra Large), XXL (Extra Extra Large).';
ALTER TABLE sales_suppliers_gold
  ALTER COLUMN longitude    COMMENT 'Geographic longitude coordinate of the supplier.';
ALTER TABLE sales_suppliers_gold
  ALTER COLUMN latitude     COMMENT 'Geographic latitude coordinate of the supplier.';
ALTER TABLE sales_suppliers_gold
  ALTER COLUMN approved     COMMENT 'Approval status of the supplier. Valid values: Y (Yes). All suppliers in this dataset are approved.';

ALTER TABLE sales_suppliers_gold ALTER COLUMN supplierID SET NOT NULL;
ALTER TABLE sales_suppliers_gold ADD CONSTRAINT pk_suppliers PRIMARY KEY (supplierID) NOT ENFORCED;

-- -----------------------------------------------------------
-- sales_franchises_gold  (FK references sales_suppliers_gold PK defined above)
-- -----------------------------------------------------------
COMMENT ON TABLE sales_franchises_gold
  IS 'Bakery franchise locations worldwide. Each row represents a unique franchise with its location, size category, and assigned supplier. Use franchiseID to join with sales_transactions_gold and media_customer_reviews_gold (use the store column to join). Some franchises have the same name but different locations, so always include city or franchiseID to uniquely identify a location. IMPORTANT: Many of the franchises do not have a supplierID that matches any record in sales_suppliers_gold when combining the tables. When querying franchise-supplier relationships, always use a LEFT JOIN to include all franchises, not an INNER JOIN.';

ALTER TABLE sales_franchises_gold
  ALTER COLUMN franchiseID COMMENT 'Unique identifier for each franchise location. Join key to sales_transactions_gold.franchiseID and media_customer_reviews_gold.ref. Multiple franchises can share the same name in different locations. Always include franchiseID and city in GROUP BY when aggregating by franchise to avoid merging distinct locations.';
ALTER TABLE sales_franchises_gold
  ALTER COLUMN name        COMMENT 'Name of the franchise location. Note: some franchise names are shared across different cities, so always include city and franchiseID to uniquely identify a location.';
ALTER TABLE sales_franchises_gold
  ALTER COLUMN city        COMMENT 'City where the franchise is located. This is the franchise city, not the customer city.';
ALTER TABLE sales_franchises_gold
  ALTER COLUMN district    COMMENT 'District or neighborhood within the city where the franchise is located.';
ALTER TABLE sales_franchises_gold
  ALTER COLUMN zipcode     COMMENT 'Postal or zip code for the franchise location.';
ALTER TABLE sales_franchises_gold
  ALTER COLUMN country     COMMENT 'Country where the franchise is located. This is the franchise country, not the customer country. Use this when asked about franchise location or revenue by country. Valid values: US, Canada, Japan, Australia, Netherlands, France, Germany, Italy, Sweden.';
ALTER TABLE sales_franchises_gold
  ALTER COLUMN size        COMMENT 'Size category of the franchise. Valid values: S (Small), M (Medium), L (Large), XL (Extra Large), XXL (Extra Extra Large).';
ALTER TABLE sales_franchises_gold
  ALTER COLUMN longitude   COMMENT 'Geographic longitude coordinate of the franchise.';
ALTER TABLE sales_franchises_gold
  ALTER COLUMN latitude    COMMENT 'Geographic latitude coordinate of the franchise.';
ALTER TABLE sales_franchises_gold
  ALTER COLUMN supplierID  COMMENT 'Foreign key to sales_suppliers_gold.supplierID. IMPORTANT: Only 27 of 48 franchises have a matching supplier. The remaining 21 have supplierIDs that do not exist in sales_suppliers_gold. Always use LEFT JOIN when joining to sales_suppliers_gold to avoid silently dropping unmatched franchises.';

ALTER TABLE sales_franchises_gold ALTER COLUMN franchiseID SET NOT NULL;
ALTER TABLE sales_franchises_gold ADD CONSTRAINT pk_franchises PRIMARY KEY (franchiseID) NOT ENFORCED;
ALTER TABLE sales_franchises_gold ADD CONSTRAINT fk_franchises_supplier FOREIGN KEY (supplierID) REFERENCES sales_suppliers_gold(supplierID) NOT ENFORCED;

-- -----------------------------------------------------------
-- sales_transactions_gold
-- -----------------------------------------------------------
COMMENT ON TABLE sales_transactions_gold
  IS 'Sales transactions for bakehouse franchise products. Each row represents a single purchase. Use customerID and franchiseID to join with customer and franchise tables.';

ALTER TABLE sales_transactions_gold
  ALTER COLUMN transactionID COMMENT 'Unique identifier for each transaction.';
ALTER TABLE sales_transactions_gold
  ALTER COLUMN customerID    COMMENT 'Foreign key to sales_customers_gold.customerID. Identifies the customer who made the purchase.';
ALTER TABLE sales_transactions_gold
  ALTER COLUMN franchiseID   COMMENT 'Foreign key to sales_franchises_gold.franchiseID. Identifies the franchise where the purchase was made.';
ALTER TABLE sales_transactions_gold
  ALTER COLUMN dateTime      COMMENT 'Timestamp of when the transaction occurred.';
ALTER TABLE sales_transactions_gold
  ALTER COLUMN product       COMMENT 'Name of the bakehouse product purchased. Valid values: Austin Almond Biscotti, Golden Gate Ginger, Orchard Oasis, Outback Oatmeal, Pearly Pies, Tokyo Tidbits.';
ALTER TABLE sales_transactions_gold
  ALTER COLUMN quantity      COMMENT 'Number of units purchased in this transaction.';
ALTER TABLE sales_transactions_gold
  ALTER COLUMN unitPrice     COMMENT 'Price per unit of the product.';
ALTER TABLE sales_transactions_gold
  ALTER COLUMN totalPrice    COMMENT 'Total price for this transaction (unitPrice * quantity). Use SUM(totalPrice) to calculate revenue or total spend. When ranking customers or franchises by performance, use SUM(totalPrice) as the metric.';
ALTER TABLE sales_transactions_gold
  ALTER COLUMN paymentMethod COMMENT 'Payment method used. Valid values: amex, mastercard, visa.';
ALTER TABLE sales_transactions_gold
  ALTER COLUMN cardNumber    COMMENT 'Card number used for the transaction.';

ALTER TABLE sales_transactions_gold ALTER COLUMN transactionID SET NOT NULL;
ALTER TABLE sales_transactions_gold ADD CONSTRAINT pk_transactions PRIMARY KEY (transactionID) NOT ENFORCED;
ALTER TABLE sales_transactions_gold ADD CONSTRAINT fk_transactions_customer FOREIGN KEY (customerID) REFERENCES sales_customers_gold(customerID) NOT ENFORCED;
ALTER TABLE sales_transactions_gold ADD CONSTRAINT fk_transactions_franchise FOREIGN KEY (franchiseID) REFERENCES sales_franchises_gold(franchiseID) NOT ENFORCED;

-- -----------------------------------------------------------
-- media_customer_reviews_gold
-- -----------------------------------------------------------
COMMENT ON TABLE media_customer_reviews_gold
  IS 'Customer reviews for bakehouse franchise locations. Each row contains free-text feedback and a sentiment classification. Use the ref column to join with sales_franchises_gold.franchiseID to obtain franchise information. The flag column contains the sentiment classification.';

ALTER TABLE media_customer_reviews_gold
  ALTER COLUMN new_id       COMMENT 'Unique identifier for each review.';
ALTER TABLE media_customer_reviews_gold
  ALTER COLUMN ref          COMMENT 'Foreign key to sales_franchises_gold.franchiseID. Identifies the franchise (store/location) being reviewed. Join this column to sales_franchises_gold.franchiseID to get franchise names and locations. Always include ref, franchise name and city in SELECT and GROUP BY when counting or aggregating reviews per franchise, since franchise names are not unique and we want exact information about the reviews by franchise.';
ALTER TABLE media_customer_reviews_gold
  ALTER COLUMN review       COMMENT 'Free-text customer review. Contains unstructured feedback about the franchise experience.';
ALTER TABLE media_customer_reviews_gold
  ALTER COLUMN review_date  COMMENT 'Date when the review was submitted.';
ALTER TABLE media_customer_reviews_gold
  ALTER COLUMN flag         COMMENT 'Sentiment classification of the review. Valid values: positive, negative, mixed. "Bad" reviews mean negative. "Neutral" reviews mean mixed. "Good or great" means positive. Use this column to filter reviews by sentiment rather than analyzing the raw review text.';

ALTER TABLE media_customer_reviews_gold ALTER COLUMN new_id SET NOT NULL;
ALTER TABLE media_customer_reviews_gold ADD CONSTRAINT pk_reviews PRIMARY KEY (new_id) NOT ENFORCED;
ALTER TABLE media_customer_reviews_gold ADD CONSTRAINT fk_reviews_franchise FOREIGN KEY (ref) REFERENCES sales_franchises_gold(franchiseID) NOT ENFORCED;

### H2. Synonyms and Display Names in the Knowledge Store

Beyond descriptions, the Knowledge Store adds two more metadata layers. 

- **Synonyms and display names**:

  - **Synonyms** - Alternate names that map business language to column names. This will help Genie map the data to the different ways users would ask the prompt. 

  - **Display names** - Uses a friendly display name for this column. **This won't rename the column in Unity Catalog.**

    **NOTE:** These will be configured in the knowledge store UI.  If using a **Metric View** as the source, they can be configured there. 

1. In your **Genie Agent**, select **Configure** > **Sources** > **sales_transactions_gold**.

2. You should see the **metadata we added to the tables in Unity Catalog**. If not, refresh your **Genie Agent**.

3. Next, view the metdata for one of the columns:

   a. Select the **pencil icon** next to the **totalPrice** column.

   b. **Display Name** field - `Total Price`

   c. **Synonyms** field, add: `revenue, sales, total sales, total revenue, income`

   d. Click **Save**.

**NOTE:** This allows Genie to map terms like "revenue" or "sales" to the **totalPrice** column.

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Note
  </strong>
  <div style="color:#333;">

For training purposes, we will stop here with one example.

Feel free to continue adding synonyms and display names to other columns as needed.

  </div>
</div>


## I. Build the Knowledge Store (SQL Expression, SQL Query, SQL Join)

### I1. Add a Region SQL Expression to the Knowledge Store

Our business groups franchise countries into three specific regions:
  - **APJ**
  - **AMER**
  - **EMEA**

This mapping does not exist as a column, we need to teach it to Genie as a SQL expression.

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Benchmark This Fixes
  </strong>
  <div style="color:#333;">

**Benchmark: What are total sales by region?** - Genie most likely groups by `country` because there is no `region` column. This expression creates the mapping.

  </div>
</div>

1. In your Genie Agent, select **Configure** > **Examples** > **Add**.

2. Select **Field**.

3. **Name the Field:** `region`

4. Run the cell below to generate your specific **SQL Expression** (adds your unique catalog name in the expression for you).
    - **NOTES:** 
        - When referencing the column in the expression, we are using a three level namespace + the column name
        - `YOUR_CATALOG.genie_course_bakehouse.sales_franchises_gold.country`

In [0]:
DECLARE OR REPLACE my_query STRING DEFAULT "";

SET VAR my_query = "
CASE
    WHEN " || my_catalog || ".genie_course_bakehouse.sales_franchises_gold.country IN ('Japan', 'Australia') 
      THEN 'APJ'
    WHEN " || my_catalog || ".genie_course_bakehouse.sales_franchises_gold.country IN ('US', 'Canada') 
      THEN 'AMER'
    WHEN " || my_catalog || ".genie_course_bakehouse.sales_franchises_gold.country IN ('Netherlands', 'France', 'Germany', 'Italy', 'Sweden')
      THEN 'EMEA' 
    ELSE 'Other'
END
";

SELECT my_query;

5. Copy the output above and paste into the **Code** field and review the query.

6. Add the following:
    - **Synonyms**: `franchise region, region sales`
    - **Instructions**: `When user asks for metrics by 'region' where region is defined by business mapping of country to APJ, AMER, EMEA, or Other; SCOPE_TABLES: Use with sales_franchises_gold and sales_transactions_gold, grouping or reporting at the franchise-location level; RISK_IF_MISUSED: Do not use unless the user explicitly refers to regions per this mapping (e.g., could mislead if a different regional breakdown is implied). This is a company-specific mapping.`

7. Select **Save**.


### I2. Add an Example SQL Query

Example SQL queries teach Genie **query patterns** that it is unlikely to generate on its own. 

Genie uses these as trusted templates when similar questions are asked.

**Add example SQL queries and functions** - [AWS](https://docs.databricks.com/aws/en/genie-agents/tune-quality#-add-example-sql-queries-and-functions) |
 [Azure](https://learn.microsoft.com/en-us/azure/databricks/genie-agents/tune-quality#-add-example-sql-queries-and-functions) | 
 [GCP](https://docs.databricks.com/gcp/en/genie-agents/tune-quality#-add-example-sql-queries-and-functions)


<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Common Question - Who are our best customers?
  </strong>
  <div style="color:#333;">

The Genie Agent will guess what "best" means, adds extra columns, and does not limit results. This instruction locks in the definition.

  </div>
</div>

Add an example query to teach Genie what query to use when someone asks "Who are the best customers".

1. In your Genie Agent, select **Configure** > **Examples**.

2. Select **+ Add** > **Example query**.

3. Enter the question: `Who are our best customers?`

4. Run the cell below to generate the query with your catalog name, then paste it as the SQL query.

In [0]:
DECLARE OR REPLACE my_query STRING DEFAULT "";

SET VAR my_query = "

SELECT
  c.customerID,
  c.first_name,
  c.last_name,
  SUM(t.totalPrice) AS total_spend
FROM
  " || my_catalog || ".genie_course_bakehouse.sales_transactions_gold AS t
    JOIN " || my_catalog || ".genie_course_bakehouse.sales_customers_gold AS c
      ON t.customerID = c.customerID
GROUP BY
  c.customerID,
  c.first_name,
  c.last_name
ORDER BY
  total_spend DESC
LIMIT 10
";

SELECT my_query;

5. Expand **Usage Guidance**.

6. Add the following: ```When users ask for the best customers make sure you aggregate by total price and return the top 10 customers.```

7. Select **Preview** to view the results.

8. Then **Save**.


### I3. Add a SQL Join

Let's view the SQL joins in the knowledge store and add one.

1. Select **Examples > Add**

2. Select **Join** 

3. Add a join:
    - Left Table: **sales_franchises_gold**
    - Right Table: **sales_suppliers_gold**.
    - Join Condition: **supplierID** on both.
    - Relationship Type: **Many to One**

4. Add the following to the **Instructions**: `Many rows in the sales_franchises_gold table have a supplierID with no match in the sales_suppliers_gold table. This is important information and make sure to display the orphaned rows.`.

5. Select **Save**.

### I4. Add General Instructions


1. In your Genie Agent, select **Configure** > **Instructions**.

2. Add the following instructions:


<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
- Formatting
    - Format all results using US dollars  
    - Round all currency values to 2 decimal places.

- Business Rules
    - `totalPrice` represents both revenue and sales volume  
    - Always include franchise name with franchise ID  
    - Use `totalPrice` for total sales and performance comparisons  

- Fiscal Calendar
    - Fiscal year starts in February  
    - Q1: February, March, April  
    - Q2: May, June, July  
    - Q3: August, September, October  
    - Q4: November, December, January  

</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>


3. Select **Save**.

## J. Re-run the Benchmarks (Test and Iterate)

Now measure the impact. Benchmarks give you a structured, repeatable way to confirm your curation actually improved accuracy, and did not break anything that already worked.

1. In your Genie Agent, select the **Benchmarks** tab.

2. Confirm you see the two benchmark questions (each with ground truth SQL):

3. Select **Run all benchmarks (Chat Mode)**.
   - Each question runs as a **new, independent conversation**. Genie carries no context between them, simulating a first-time user.

4. As each completes, review the **Assessment**, **Failure analysis**, and **Result match**. Both benchmarks should pass with additional context in the Knowledge Store of the Genie Agent.

<div style="max-width: 1150px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<!-- HEADER BAND -->
<div style="background: #1B5162; color: white; border-radius: 10px 10px 4px 4px; padding: 20px 26px; margin-bottom: 16px;">
  <div style="font-size: 12pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: #B7CDD5; margin-bottom: 6px;">Where To Go From Here</div>
  <div style="font-size: 21pt; font-weight: 700; line-height: 1.25;">This is a starting point, not production</div>
  <div style="font-size: 15pt; opacity: 0.92; margin-top: 6px; line-height: 1.45;">You added a few high-impact pieces of context and saw the benchmarks improve. A dependable Genie Agent is a <strong>product you keep curating</strong>, not a one-time build.</div>
</div>

<!-- THREE CARDS -->
<div style="display: flex; gap: 16px; flex-wrap: nowrap; margin-bottom: 16px;">

  <!-- CARD 1 -->
  <div style="flex: 1; background: #F9F7F4; border-radius: 8px; padding: 20px 22px; position: relative; box-shadow: 0 2px 8px rgba(27,49,57,0.06); box-sizing: border-box;">
    <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #4299E0; border-radius: 8px 8px 0 0;"></div>
    <div style="font-size: 16pt; font-weight: 700; margin: 8px 0 10px 0;">Grow your benchmark suite</div>
    <ul style="margin: 0; padding-left: 20px; font-size: 14pt; line-height: 1.6;">
      <li style="margin-bottom: 8px;">We added <strong>two</strong>. Real spaces have <strong>many more</strong></li>
      <li style="margin-bottom: 8px;">Cover simple lookups, aggregations, ambiguous terms, and data-quality edge cases</li>
      <li style="margin-bottom: 8px;">Treat them as your <strong>acceptance criteria</strong></li>
      <li>Re-run them on <strong>every change</strong></li>
    </ul>
  </div>

  <!-- CARD 2 -->
  <div style="flex: 1; background: #F9F7F4; border-radius: 8px; padding: 20px 22px; position: relative; box-shadow: 0 2px 8px rgba(27,49,57,0.06); box-sizing: border-box;">
    <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #00A972; border-radius: 8px 8px 0 0;"></div>
    <div style="font-size: 16pt; font-weight: 700; margin: 8px 0 10px 0;">Keep adding context</div>
    <ul style="margin: 0; padding-left: 20px; font-size: 14pt; line-height: 1.6;">
      <li style="margin-bottom: 8px;">Cover the questions your users <strong>actually ask</strong></li>
      <li style="margin-bottom: 8px;">More <strong>descriptions</strong> and <strong>synonyms</strong></li>
      <li>Add <strong>SQL expressions</strong>, <strong>example queries</strong>, and <strong>joins</strong> across all your tables</li>
    </ul>
  </div>

  <!-- CARD 3 -->
  <div style="flex: 1; background: #F9F7F4; border-radius: 8px; padding: 20px 22px; position: relative; box-shadow: 0 2px 8px rgba(27,49,57,0.06); box-sizing: border-box;">
    <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #FFAB00; border-radius: 8px 8px 0 0;"></div>
    <div style="font-size: 16pt; font-weight: 700; margin: 8px 0 10px 0;">Iterate from real use</div>
    <ul style="margin: 0; padding-left: 20px; font-size: 14pt; line-height: 1.6;">
      <li style="margin-bottom: 8px;">Review <strong>Monitoring</strong> feedback</li>
      <li style="margin-bottom: 8px;">Turn thumbs-down and failures into <strong>new benchmarks</strong> and fixes</li>
      <li style="margin-bottom: 8px;">Validate definitions with <strong>subject matter experts</strong></li>
      <li><strong>Unity Catalog</strong> keeps access governed throughout</li>
    </ul>
  </div>

</div>

<!-- TAKEAWAY -->
<div style="background: #FFF6F4; border-left: 4px solid #FF5F46; border-radius: 6px; padding: 14px 20px; font-size: 14pt; line-height: 1.55;">
  <span style="font-size: 12pt; font-weight: 700; letter-spacing: 1.5px; color: #B83A24; text-transform: uppercase; display: block; margin-bottom: 4px;">Not production-ready yet</span>
  This demo showed the <strong>method</strong> on a handful of examples. Before your team relies on this agent, expand the benchmarks, add context for the questions people really ask, and keep iterating from monitoring. Curation is continuous: every real user question is a chance to improve the space.
</div>

<!-- DOCS CALLOUT -->
<div style="margin-top: 16px; padding: 16px 24px; background: #1B5162; border-radius: 8px; display: flex; align-items: center; justify-content: space-between; gap: 20px;">
  <div style="font-size: 15pt; color: white; line-height: 1.5;">
    For official guidance on improving a Genie Agent, see the <strong>tune response quality</strong> documentation.
  </div>
  <div style="display: flex; gap: 10px; flex-shrink: 0;">
    <a href="https://docs.databricks.com/aws/en/genie-agents/tune-quality" target="_blank" style="background: white; color: #1B5162; font-size: 14pt; font-weight: 700; padding: 10px 20px; border-radius: 6px; text-decoration: none; white-space: nowrap;">AWS →</a>
    <a href="https://learn.microsoft.com/en-us/azure/databricks/genie-agents/tune-quality" target="_blank" style="background: white; color: #1B5162; font-size: 14pt; font-weight: 700; padding: 10px 20px; border-radius: 6px; text-decoration: none; white-space: nowrap;">Azure →</a>
    <a href="https://docs.databricks.com/gcp/en/genie-agents/tune-quality" target="_blank" style="background: white; color: #1B5162; font-size: 14pt; font-weight: 700; padding: 10px 20px; border-radius: 6px; text-decoration: none; white-space: nowrap;">GCP →</a>
  </div>
</div>

</div>


## K. What We Did Not Cover

This workshop focused on the core Knowledge Store curation tools: descriptions, synonyms, prompt matching, SQL expressions, example queries, text instructions, and benchmarks. A few additional capabilities are worth exploring.

<br></br>

<div style="max-width: 1200px; margin: 0 auto; font-family: sans-serif;">
<div style="display: flex; gap: 20px; justify-content: center; flex-wrap: wrap;">

<div style="flex: 1; min-width: 320px; max-width: 380px; background: #F9F7F4; border-radius: 8px; padding: 18px; position: relative; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
  <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #00A972; border-radius: 8px 8px 0 0;"></div>
  <div style="font-size: 18pt; font-weight: 700; color: #0b2026; margin-bottom: 8px; margin-top: 6px;">Trusted Assets</div>
  <div style="font-weight: 600; font-size: 14pt; margin-bottom: 4px;">What it is</div>
  <ul style="padding-left: 18px; margin: 0 0 8px 0; font-size: 14pt; color: #333; line-height: 1.5;">
    <li>SQL functions registered in Unity Catalog</li>
    <li>Genie calls them directly to answer questions</li>
    <li>Responses labeled <strong>"Trusted"</strong></li>
  </ul>
  <div style="font-weight: 600; font-size: 14pt; margin-bottom: 4px;">Why it matters</div>
  <ul style="padding-left: 18px; margin: 0; font-size: 14pt; color: #555; line-height: 1.4;">
    <li>Guaranteed, repeatable answers</li>
    <li>Ideal for KPIs, compliance, executive reporting</li>
  </ul>
  <div style="margin-top: 12px; background: rgba(0,169,114,0.10); border-left: 4px solid #00A972; padding: 8px 10px; border-radius: 6px; font-size: 14pt;">
    <strong>Docs:</strong> <a href="https://docs.databricks.com/aws/en/genie/add-instructions#trusted-assets" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/genie-agents/tune-quality#trusted-assets" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/genie/add-instructions#trusted-assets" target="_blank">GCP</a>
  </div>
</div>

<div style="flex: 1; min-width: 320px; max-width: 380px; background: #F9F7F4; border-radius: 8px; padding: 18px; position: relative; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
  <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #4299E0; border-radius: 8px 8px 0 0;"></div>
  <div style="font-size: 18pt; font-weight: 700; color: #0b2026; margin-bottom: 8px; margin-top: 6px;">Parameterized Queries</div>
  <div style="font-weight: 600; font-size: 14pt; margin-bottom: 4px;">What it is</div>
  <ul style="padding-left: 18px; margin: 0 0 8px 0; font-size: 14pt; color: #333; line-height: 1.5;">
    <li>Queries with named parameters like <code>:limit</code>, <code>:country</code></li>
    <li>One query supports many variations</li>
    <li>Examples: top 3, top 10, top in Japan</li>
  </ul>
  <div style="font-weight: 600; font-size: 14pt; margin-bottom: 4px;">Why it matters</div>
  <ul style="padding-left: 18px; margin: 0; font-size: 14pt; color: #555; line-height: 1.4;">
    <li>Reduces the number of example queries needed</li>
    <li>Still produces <strong>"Trusted"</strong> responses when used directly</li>
  </ul>
  <div style="margin-top: 12px; background: rgba(66,153,224,0.10); border-left: 4px solid #4299E0; padding: 8px 10px; border-radius: 6px; font-size: 14pt;">
    <strong>Docs:</strong> <a href="https://docs.databricks.com/aws/en/genie/add-instructions#add-query-parameters" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/genie-agents/tune-quality#trusted-assets" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/genie/add-instructions#add-query-parameters" target="_blank">GCP</a>
  </div>
</div>

<div style="flex: 1; min-width: 320px; max-width: 380px; background: #F9F7F4; border-radius: 8px; padding: 18px; position: relative; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
  <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #FFAB00; border-radius: 8px 8px 0 0;"></div>
  <div style="font-size: 18pt; font-weight: 700; color: #0b2026; margin-bottom: 8px; margin-top: 6px;">Metric Views</div>
  <div style="font-weight: 600; font-size: 14pt; margin-bottom: 4px;">What it is</div>
  <ul style="padding-left: 18px; margin: 0 0 8px 0; font-size: 14pt; color: #333; line-height: 1.5;">
    <li>Code-managed semantic layer in Unity Catalog</li>
    <li>Defined using YAML</li>
    <li>Includes metrics, dimensions, synonyms, formatting</li>
  </ul>
  <div style="font-weight: 600; font-size: 14pt; margin-bottom: 4px;">Why it matters</div>
  <ul style="padding-left: 18px; margin: 0; font-size: 14pt; color: #555; line-height: 1.4;">
    <li>Shared across Genie and AI/BI dashboards</li>
    <li>Ensures consistent metric definitions</li>
  </ul>
  <div style="margin-top: 12px; background: rgba(255,171,0,0.15); border-left: 4px solid #FFAB00; padding: 8px 10px; border-radius: 6px; font-size: 14pt;">
    <strong>Docs:</strong> <a href="https://docs.databricks.com/aws/en/metric-views/" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/metric-views/" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/metric-views/" target="_blank">GCP</a>
  </div>
</div>

</div>
</div>

## L. Conclusion

In this notebook, you:

1. Reviewed common **Genie Agent failure modes**.
2. **Benchmarked** the un-curated agent to establish a baseline and saw the region question fail.
3. Learned the **curation mindset** and the **Knowledge Store order of operations**.
4. Added table and column **metadata** in Unity Catalog with `ALTER TABLE`.
5. **Added context to the Knowledge Store**: a region SQL expression, an example query, a franchise-to-supplier join, and general instructions.
6. **Re-ran the benchmarks** to measure the improvement.

The priority holds: **metadata first**, then **SQL logic** for business definitions, and **text instructions** only for what SQL cannot express. Curation is iterative. Every real user question is a chance to improve the space.


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>